# Contingency Analysis in Transmission Grids

Contingency analysis is a critical process in power system operations used to assess the impact of potential failures (e.g., line outages) on grid stability and reliability. It helps operators prepare for unexpected events by simulating scenarios such as N-1 or N-2 contingencies, where one or more components are removed from service. This analysis ensures that the grid can continue to operate within safe limits even under stressed conditions.

---

## Dataset Generation and Model Evaluation

The dataset used in this study originates from the Texas transmission grid, which includes approximately 2,000 nodes. Using the contingency mode of the `gridfm-datakit`, we simulated N-2 contingencies by removing up to two transmission lines at a time. For each scenario, we first solved the optimal power flow (OPF) problem to determine the generation dispatch. Then, we applied the contingency by removing lines and re-solved the power flow to observe the resulting grid state.

This process generated around 100,000 unique scenarios. Our model, **GENCO**, was trained on this dataset to predict power flow outcomes. For demonstration purposes, we selected a subsample of 181 scenarios. The `gridfm-datakit` also computed DC power flow results, enabling a comparison between GENCO predictions and traditional DC power flow estimates, specifically in terms of line loading accuracy.

All predictions are benchmarked against the ground truth obtained from AC power flow simulations. Additionally, we analyze bus voltage violations, which GridFM can predict but are not captured by the DC solver, highlighting GENCO’s enhanced capabilities in modeling grid behavior.


In [ ]:
# Only run this cell if running on Colab.

import sys

if "google.colab" in sys.modules:
    try:
        #!git clone https://github.com/gridfm/gridfm-graphkit.git   CHANGE IF WE PUT THIS NOTEBOOK ON MAIN
        !git clone -b normalization_fixed_eval_predict https://github.com/gridfm/gridfm-graphkit.git
        !pip install ./gridfm-graphkit
    except Exception as e:
        print(f"Failed to start Google Collab setup, due to {e}")

# If running on Colab you will be asked to restart the session. This is normal and you should do so.


In [ ]:
%cd gridfm-graphkit/examples/notebooks/
%pwd

In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from contingency_utils import *
import glob

## Load Data

We load both the ground truth and predicted values of the power flow solution. The predictions are generated using the `gridfm-graphkit` CLI:

```bash
gridfm-graphkit predict ...
```

We then merge the datasets using `scenario` and `bus` as keys, allowing us to align the predicted and actual values for each grid state and bus.

In [4]:
root_pred_folder = "../data/k_2/"
prediction_dir = "test_lineloadings"
label_plot = "GENCO_Texas"

In [4]:
preds = pd.read_parquet(os.path.join(root_pred_folder, "preds/Texas2k_case1_2016summerpeak_predictions.parquet"))
bus_data = pd.read_parquet(os.path.join(root_pred_folder, "raw/bus_data.parquet/da40e377c2264d5c86a668c9e28ca566-0.parquet"))
branch_data = pd.read_parquet(os.path.join(root_pred_folder, "raw/branch_data.parquet/7d82df9ae566416c80af53c81e1aac68-0.parquet"))

In [8]:
# n-2 on ccc
preds = pd.read_parquet("/dccstor/gridfm/mlflow_alban_contingency/504276166362365184/45509e8f71a3455689720d8e7a5b259b/artifacts/test/Texas2k_case1_2016summerpeak_predictions.parquet")
bus_data = pd.read_parquet("/dccstor/gridfm/powermodels_data/v4/texas_contingency/test_2/Texas2k_case1_2016summerpeak/raw/bus_data.parquet/scenario_partition=0/da40e377c2264d5c86a668c9e28ca566-0.parquet")
branch_data = pd.read_parquet("/dccstor/gridfm/powermodels_data/v4/texas_contingency/test_2/Texas2k_case1_2016summerpeak/raw/branch_data.parquet/scenario_partition=0/7d82df9ae566416c80af53c81e1aac68-0.parquet")

In [ ]:
# Filter data to use same scenarios
bus_scenarios = bus_data['scenario'].values
filtered_preds = preds[preds['scenario'].isin(bus_scenarios)]
preds_scenarios = filtered_preds['scenario'].values
filtered_bus_data = bus_data[bus_data['scenario'].isin(preds_scenarios)]
filtered_branch_data = branch_data[branch_data['scenario'].isin(preds_scenarios)] 

In [ ]:
# Create one df for GT and preds
pf_node = preds.merge(combined_df, on=["scenario", "bus"], how="left")
pf_node.head()

In [ ]:
# Correct Vm and Va values for PV and REF buses
pf_node["Vm_pred_corrected"] = pf_node["vm_pu"]
pf_node["Va_pred_corrected"] = pf_node["va"]
pf_node.loc[pf_node.PV == 1, "Vm_pred_corrected"] = pf_node.loc[pf_node.PV == 1, "Vm"]
pf_node.loc[pf_node.REF == 1, "Va_pred_corrected"] = pf_node.loc[pf_node.REF == 1, "Va"]

pf_node["Vm_dc"] = 1 # Create column set values to 1

pf_node["Vm_dc_corrected"] = pf_node["Vm_dc"]
pf_node["Va_dc_corrected"] = pf_node["Va_dc"]

pf_node.loc[pf_node.PV == 1, "Vm_dc_corrected"] = pf_node.loc[pf_node.PV == 1, "Vm"]
pf_node.loc[pf_node.REF == 1, "Va_dc_corrected"] = pf_node.loc[pf_node.REF == 1, "Va"]

In [ ]:
# All angles in degrees
pf_node['Va_pred_corrected'] = np.rad2deg(pf_node['Va_pred_corrected'])
pf_node['va_target'] = np.rad2deg(pf_node['va_target'])
pf_node

## Compute branch current and line loading

In [ ]:
#### Create loadings

loadings = []
loadings_pred = []
loadings_dc = []

results = {}

for scenario_idx in tqdm(pf_node.scenario.unique()):
    pf_node_scenario = pf_node[pf_node.scenario == scenario_idx]
    #branch_data_scenario = filtered_branch_data[filtered_branch_data.scenario == scenario_idx]
    branch_data_scenario = combined_branch_df[combined_branch_df.scenario == scenario_idx]

    flags = ['gt', 'pred', 'dc']
    for f in flags:
        
        # Get power flows - of all branches that are 0 and 1
        pf, qf, pt, qt = compute_branch_powers_vectorized(branch_data_scenario, pf_node_scenario, sn_mva=100.0, flag=f)

        results[f] = {
        'pf': pf,
        'qf': qf,
        'pt': pt,
        'qt': qt
        }
        
        s_from = np.sqrt(results[f]['pf'] ** 2  + results[f]['qf'] ** 2)
        s_to = np.sqrt(results[f]['pt']** 2  + results[f]['qt'] ** 2)

        #branch_removed_idx = branch_data_scenario[branch_data_scenario['br_status']==0].idx.values.astype(int)
        
        rated_branches = branch_data_scenario[(branch_data_scenario["rate_a"] > 0)].copy()
        rate_a = rated_branches["rate_a"].to_numpy()  

        # check if line flow is 0
        #print(s_from[branch_removed_idx])

        # Loading = max(S_from, S_to) / rate_a
        loading = np.maximum(s_from, s_to) / rate_a
    
        if f == "gt":
            loadings.append(loading)
        elif f == "pred":
            loadings_pred.append(loading)
        elif f == "dc": 
            loadings_dc.append(loading)

loadings = np.array(loadings)
loadings_pred = np.array(loadings_pred)
loadings_dc = np.array(loadings_dc)


In [11]:
loadings = loadings.flatten()
loadings_pred = loadings_pred.flatten()
loadings_dc = loadings_dc.flatten()

overloadings_mask = (loadings > 0.95)
overloadings_pred_mask = (loadings_pred > 0.95)
overloadings_dc_mask = (loadings_dc > 0.95)

In [12]:
overloadings = loadings[overloadings_mask == True]
overloadings_pred = loadings_pred[overloadings_pred_mask == True]
overloadings_dc = loadings_dc[overloadings_dc_mask == True]

## Compute metrics for overloading classification for GridFM and DC PF
- Below are the results of GENCO for overloading classification (HGNS version)
- Ground truth is AC PF.

In [ ]:
TP_genco, FP_genco, TN_genco, FN_genco = compute_cm_metrics(
    overloadings_mask, overloadings_pred_mask, prediction_dir, label_plot
)

- Below are the results of DC PF for overloading classification
- Ground truth is AC PF.
- We use a threshold of 0.95 to make sure we identify all overloads

In [ ]:
TP_dc, FP_dc, TN_dc, FN_dc = compute_cm_metrics(
    overloadings_mask, overloadings_dc_mask, "DC", label_plot
)

## Histogram of true line loadings

In [ ]:
plt.hist(loadings, bins=100)
plt.xlabel("Line Loadings")
plt.ylabel("Frequency")
plt.title("Line loadings")
# log scale
plt.savefig(f"loadings_histogram_{prediction_dir}.png")
plt.show()

## Predicted vs True line loading

In [ ]:
true_vals = loadings
gfm_vals = loadings_pred
dc_vals = loadings_dc

plot_mass_correlation_density(true_vals, gfm_vals, prediction_dir, label_plot)

In [ ]:
plot_mass_correlation_density(true_vals, dc_vals, "DC", "DC Solver")

In [27]:
plot_cm(TN_genco, FP_genco, FN_genco, TP_genco, prediction_dir, label_plot)

In [ ]:
plot_cm(TN_dc, FP_dc, FN_dc, TP_dc, "DC", "DC Solver")

In [ ]:
# Histograms of loadings
plot_loading_predictions(
    loadings_pred,
    loadings_dc,
    loadings,
    prediction_dir,
    label_plot,
)

In [ ]:
# make bar plot of wrongly classified loadings for different bins
bins = np.arange(0, 2.2, 0.2)
mse_pred = []
mse_dc = []
for i in range(len(bins) - 1):
    idx_in_bins = (loadings > bins[i]) & (
        loadings< bins[i + 1]
    )
    mse_pred.append(
        np.mean(
            (
                loadings_pred[idx_in_bins]
                - loadings[idx_in_bins]
            )
            ** 2
        )
    )
    mse_dc.append(
        np.mean(
            (
                loadings_dc[idx_in_bins]
                - loadings[idx_in_bins]
            )
            ** 2
        )
    )
# labels
labels = [f"{bins[i]:.1f}-{bins[i + 1]:.1f}" for i in range(len(bins) - 1)]
plt.bar(labels, mse_pred, label=label_plot, alpha=0.5)
plt.bar(labels, mse_dc, label="DC", alpha=0.5)
plt.legend()
plt.xlabel("Loadings")
plt.ylabel("MSE")
# y log scale
plt.yscale("log")
# rotate x labels
plt.xticks(rotation=45)
plt.savefig(f"loading_mse_{prediction_dir}.png")
plt.show()

## Voltage violations

In [32]:
plot_mass_correlation_density_voltage(pf_node, prediction_dir, label_plot)

In [ ]:
# Compute CM for Vm and Vm predicted
vm_violation_mask = (pf_node["Vm"] > 1.00)
vm_pred_violation_mask = (pf_node["Vm_pred_corrected"] > 1.00)

In [34]:
TP_genco_vm, FP_genco_vm, TN_genco_vm, FN_genco_vm = compute_cm_metrics(
    vm_violation_mask, vm_pred_violation_mask, "Vm violations", label_plot
)

In [ ]:
plot_cm(TN_genco_vm, FP_genco_vm, TP_genco_vm, TP_dc, "VM", "Vm Violation")
